# GraviLens v2 driver

Thin Colab driver only. All simulation, training, evaluation, and audit logic lives in the `gravilens` package.

Set `REPO` below to the public repository URL. A GPU runtime is strongly recommended for the ablation; data generation is CPU-bound.

In [ ]:
import os
REPO = "https://github.com/<handle>/gravilens"
if not os.path.isdir('/content/gravilens'):
    !git clone {REPO} /content/gravilens
%cd /content/gravilens
!pip -q install -e ".[sim,dev]"

In [ ]:
import torch
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## Generate the matched dataset

The Galaxy10 DECals file is downloaded once (~2.5 GB). The full 3,333 triplets cost roughly 5 CPU-hours single-process; the three splits can run concurrently. Shards are resumable, so rerunning the cell continues where it stopped. Skip this cell if `data/gravilens-v2` already exists.

In [ ]:
!python -m gravilens generate --config configs/data.yaml

## Smoke test (2 epochs)

This checks the full train/evaluate path before spending GPU time on the 21-run grid.

In [ ]:
!python -m gravilens train --config configs/ablation.yaml --model gravilens --variant full --seed 42 --epochs 2
!python -m gravilens evaluate --run runs/gravilens_full_seed42

## Full ablation (21 runs)

4 variants × 3 seeds + 3 parameter-matched baselines × 3 seeds. `ablate` skips runs that already have `metrics.json`. `audit` exits non-zero until all runs exist; `figures` writes the main table and figures from the artifacts only.

In [ ]:
!python -m gravilens ablate --config configs/ablation.yaml
!python -m gravilens audit
!python -m gravilens figures

## Save artifacts

Colab runtimes are ephemeral. Copy `runs/` to Drive or download the run directories before disconnecting.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r runs /content/drive/MyDrive/gravilens-runs